[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/05_finetune_and_compare.ipynb)

# Step 5 — Fine-Tune and Compare (Optional Demo)

LoRA fine-tune a small model on the grounded synthetic corpus, then re-run the Step 1 test set.

## Learning objectives
- Prepare an instruction dataset for SFT
- Fine-tune with TRL + PEFT (4-bit LoRA)
- Compare baseline vs fine-tuned scores by failure mode

## Prerequisit:
1. Generated synthetic_train.jsonl and test_set.jsonl files
2. Install SFT dependencies:
    ``` bash
    uv sync --group text-sft
    ```
3. Set ``RUN_SFT`` to 1 to run GPU LoRA fine-tuning in .env (needs CUDA; not via Ollama)

In [ ]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SCORES_PATH,
    COMPARISON_REPORT_PATH,
    FAILURE_MODE_GUIDANCE,
    FINETUNED_PREDICTIONS_PATH,
    SYNTHETIC_IBT_PATH,
    TEST_SET_PATH,
    FailureMode,
    QASample,
    build_sft_dataset,
    compare_summaries,
    create_judge_client,
    create_small_model_client,
    load_implementation_dotenv,
    load_typed_jsonl,
    qa_samples_to_messages,
    read_json,
    read_jsonl,
    run_inference,
    save_baseline_results,
    score_predictions,
    train_lora_sft,
    use_repo_root,
    write_json,
)
from aieng.syn_data.text.sft import PeftInferenceClient
from rich.console import Console
from rich.table import Table


load_implementation_dotenv()
ROOT = use_repo_root(Path("."))

MODELS_DIR = ROOT / "implementations" / "qa_text_generation" / "models" / "lora_adapter"
BASE_MODEL = os.getenv("SFT_BASE_MODEL", "Qwen/Qwen2.5-3B-Instruct")
RUN_SFT = os.getenv("RUN_SFT", "0") == "1"

console = Console(width=100)

## 1. Load training and test data

In [ ]:
train_samples = load_typed_jsonl(SYNTHETIC_IBT_PATH, QASample.from_dict)
test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)
print(len(train_samples), len(test_samples))

36 60


## 2. LoRA SFT with TRL

Suggested base model: `Qwen/Qwen2.5-3B-Instruct` with 4-bit quantization.

In [12]:
sft_dataset = build_sft_dataset(train_samples)
print(sft_dataset)
console.print("[bold]Example training row:[/bold]")
qa_samples_to_messages(train_samples[:1])[0]

if RUN_SFT:
    adapter_path = train_lora_sft(
        train_samples,
        MODELS_DIR,
        base_model=BASE_MODEL,
        num_train_epochs=1.0,
    )
    console.print(f"[bold green]LoRA adapter saved to {adapter_path}[/bold green]")
else:
    console.print(
        "[bold yellow]\N{WARNING SIGN} SFT SKIPPED[/bold yellow]\n"
        "[italic]Set [green]RUN_SFT=1[/green] and run on a [green]CUDA[/green] machine to fine-tune.[/italic]\n"
        "[cyan]Fine-tuned evaluation will reuse the small model client.[/cyan]"
    )

Dataset({
    features: ['messages'],
    num_rows: 36
})


Example training row:

Loading weights:   0%|          | 1/434 [00:00<01:11,  6.06it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 221.34it/s]
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingfa

Step,Training Loss


LoRA adapter saved to 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/models/lora_adapter

## 3. Re-evaluate on the held-out test set

In [ ]:
judge = create_judge_client()

if RUN_SFT and MODELS_DIR.exists():
    finetuned_client = PeftInferenceClient(MODELS_DIR, BASE_MODEL)
else:
    finetuned_client = create_small_model_client()

finetuned_predictions = run_inference(finetuned_client, test_samples)
finetuned_scores = score_predictions(judge, test_samples, finetuned_predictions)

if RUN_SFT:
    eval_label = "Finetuned"
    predictions_path = FINETUNED_PREDICTIONS_PATH
    scores_path = COMPARISON_REPORT_PATH.parent / "finetuned_scores.json"
else:
    eval_label = "Baseline"
    predictions_path = BASELINE_PREDICTIONS_PATH
    scores_path = BASELINE_SCORES_PATH

finetuned_summary = save_baseline_results(
    finetuned_predictions,
    finetuned_scores,
    test_samples,
    predictions_path=predictions_path,
    scores_path=scores_path,
)

table = Table(title=f"{eval_label} Model Evaluation Summary")
table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="magenta")

for metric, score in finetuned_summary["overall"].items():
    table.add_row(metric, f"{score:.3f}")

console.print(table)

   Finetuned Model Evaluation    
             Summary             
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric                ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ correctness           │ 4.758 │
│ coherence             │ 4.983 │
│ instruction_following │ 4.883 │
│ factual_plausibility  │ 4.892 │
│ average               │ 4.879 │
└───────────────────────┴───────┘

## 4. Compare before vs after

In [ ]:
baseline_report = read_json(BASELINE_SCORES_PATH)
comparison_label = "Finetuned" if RUN_SFT else "Baseline (SFT skipped)"
comparison = {
    "baseline": baseline_report["overall"],
    "finetuned": finetuned_summary["overall"],
    "delta": compare_summaries(
        baseline_report["overall"],
        finetuned_summary["overall"],
    ),
    "by_failure_mode": {
        "baseline": baseline_report.get("by_failure_mode", {}),
        "finetuned": finetuned_summary.get("by_failure_mode", {}),
    },
    "run_sft": RUN_SFT,
    "comparison_label": comparison_label,
}
write_json(COMPARISON_REPORT_PATH, comparison)


def print_comparison_table(comparison):
    """Print a baseline vs post-SFT (or skipped) evaluation metrics table."""
    second_label = comparison.get("comparison_label", "Finetuned")
    title = (
        "Baseline vs Finetuned Model Evaluation Summary"
        if comparison.get("run_sft")
        else "Baseline Evaluation Summary (SFT skipped)"
    )
    table = Table(title=title)
    table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
    table.add_column("Baseline", justify="right", style="yellow")
    table.add_column(second_label, justify="right", style="green")
    table.add_column("Delta", justify="right", style="magenta")

    for metric in comparison["baseline"]:
        base_val = comparison["baseline"].get(metric, None)
        finetuned_val = comparison["finetuned"].get(metric, None)
        delta_val = comparison["delta"].get(metric, None)
        base_str = f"{base_val:.3f}" if isinstance(base_val, (int, float)) else str(base_val)
        finetuned_str = f"{finetuned_val:.3f}" if isinstance(finetuned_val, (int, float)) else str(finetuned_val)
        delta_str = f"{delta_val:+.3f}" if isinstance(delta_val, (int, float)) else str(delta_val)
        table.add_row(metric, base_str, finetuned_str, delta_str)

    console.print(table)


print_comparison_table(comparison)

     Baseline vs Finetuned Model Evaluation Summary      
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                ┃ Baseline ┃ Finetuned ┃  Delta ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ correctness           │    4.467 │     4.758 │ +0.292 │
│ coherence             │    4.950 │     4.983 │ +0.033 │
│ instruction_following │    4.850 │     4.883 │ +0.033 │
│ factual_plausibility  │    4.650 │     4.892 │ +0.242 │
│ average               │    4.729 │     4.879 │ +0.150 │
└───────────────────────┴──────────┴───────────┴────────┘

## 5. Look back at targeted failure modes

Step 1 did not sample generic Q&A. Each held-out item was generated to stress one of four small-model weaknesses:

| Failure mode | Document role | What the test item is trying to catch |
|---|---|---|
| `format_non_compliance` | Policy-dense + scope-boundary | Ignores requested structure (JSON, numbered list, clause citation) |
| `domain_vocabulary_drift` | Policy-dense | Paraphrases away terms like APR, grace period, fiduciary |
| `multi_constraint_collapse` | Policy-dense | Answers only one of several policy rules in the question |
| `refusal_calibration` | Scope-boundary | Answers out-of-scope advice, or refuses an in-scope policy question |

Overall averages can hide a mix of wins and losses. The next cell compares **baseline vs post-SFT judge scores on those same buckets**, then shows one question per mode so you can see whether the answer actually changed.

In [ ]:
from collections import Counter

from rich.panel import Panel
from rich.text import Text


def _mode_key(sample: QASample) -> str:
    return sample.failure_mode.value if sample.failure_mode else "unknown"


def _fmt_score(value, *, signed: bool = False) -> str:
    if not isinstance(value, (int, float)):
        return "—"
    return f"{value:+.3f}" if signed else f"{value:.3f}"


mode_counts = Counter(_mode_key(sample) for sample in test_samples)
baseline_by_mode = comparison["by_failure_mode"]["baseline"]
finetuned_by_mode = comparison["by_failure_mode"]["finetuned"]
targeted_modes = [mode.value for mode in FailureMode] + [
    key for key in set(baseline_by_mode) | set(finetuned_by_mode) if key not in {mode.value for mode in FailureMode}
]

failure_mode_deltas = {}
for mode in targeted_modes:
    if mode not in baseline_by_mode and mode not in finetuned_by_mode:
        continue
    failure_mode_deltas[mode] = compare_summaries(
        baseline_by_mode.get(mode, {}),
        finetuned_by_mode.get(mode, {}),
    )

comparison["by_failure_mode"]["delta"] = failure_mode_deltas
write_json(COMPARISON_REPORT_PATH, comparison)

table = Table(title="Targeted failure modes: baseline vs post-SFT")
table.add_column("Failure mode", style="cyan")
table.add_column("n", justify="right")
table.add_column("What Step 1 targeted")
table.add_column("Baseline avg", justify="right", style="yellow")
table.add_column(comparison_label + " avg", justify="right", style="green")
table.add_column("Δ average", justify="right", style="magenta")
table.add_column("Δ correctness", justify="right")
table.add_column("Δ instruction following", justify="right")
table.add_column("Δ factual plausibility", justify="right")

for mode in targeted_modes:
    if mode not in failure_mode_deltas:
        continue
    try:
        guidance = FAILURE_MODE_GUIDANCE[FailureMode(mode)]
    except (KeyError, ValueError):
        guidance = "Present in the test set but not one of the four generation targets."
    table.add_row(
        mode,
        str(mode_counts.get(mode, 0)),
        guidance,
        _fmt_score(baseline_by_mode.get(mode, {}).get("average")),
        _fmt_score(finetuned_by_mode.get(mode, {}).get("average")),
        _fmt_score(failure_mode_deltas[mode].get("average"), signed=True),
        _fmt_score(failure_mode_deltas[mode].get("correctness"), signed=True),
        _fmt_score(failure_mode_deltas[mode].get("instruction_following"), signed=True),
        _fmt_score(failure_mode_deltas[mode].get("factual_plausibility"), signed=True),
    )

console.print(table)

improved = [
    mode
    for mode, delta in failure_mode_deltas.items()
    if isinstance(delta.get("average"), (int, float)) and delta["average"] > 0.02
]
regressed = [
    mode
    for mode, delta in failure_mode_deltas.items()
    if isinstance(delta.get("average"), (int, float)) and delta["average"] < -0.02
]
unchanged = [mode for mode in failure_mode_deltas if mode not in improved and mode not in regressed]
console.print(
    Panel(
        f"[green]Improved[/green]: {', '.join(improved) or 'none'}\n"
        f"[red]Regressed[/red]: {', '.join(regressed) or 'none'}\n"
        f"[dim]Roughly unchanged (|Δ| ≤ 0.02)[/dim]: {', '.join(unchanged) or 'none'}\n\n"
        "A higher average means the judge scored answers closer to the gold on that "
        "failure-mode slice. Small n per bucket — treat deltas as directional, not a leaderboard.",
        title="Did the targeted weaknesses move?",
    )
)

baseline_predictions_by_id = {}
if BASELINE_PREDICTIONS_PATH.exists():
    baseline_predictions_by_id = {row["id"]: row for row in read_jsonl(BASELINE_PREDICTIONS_PATH)}
finetuned_predictions_by_id = {row["id"]: row for row in finetuned_predictions}

samples_by_mode: dict[str, list[QASample]] = {}
for sample in test_samples:
    samples_by_mode.setdefault(_mode_key(sample), []).append(sample)

console.print(
    "\n[bold]Qualitative spot-check[/bold] — one item per targeted mode "
    "(prefer cases where the two model answers differ)."
)

for mode in [m.value for m in FailureMode]:
    candidates = samples_by_mode.get(mode, [])
    if not candidates:
        continue
    chosen = None
    for sample in candidates:
        before = baseline_predictions_by_id.get(sample.id, {}).get("model_answer", "")
        after = finetuned_predictions_by_id.get(sample.id, {}).get("model_answer", "")
        if before and after and before.strip() != after.strip():
            chosen = sample
            break
    if chosen is None:
        chosen = candidates[0]
    before = baseline_predictions_by_id.get(chosen.id, {}).get(
        "model_answer", "[baseline prediction missing — re-run notebook 01]"
    )
    after = finetuned_predictions_by_id.get(chosen.id, {}).get("model_answer", "[fine-tuned prediction missing]")
    body = Text()
    body.append("Question\n", style="bold")
    body.append(chosen.question.strip() + "\n\n")
    body.append("Gold\n", style="bold")
    body.append(chosen.gold_answer.strip()[:600] + "\n\n")
    body.append("Baseline\n", style="bold yellow")
    body.append(str(before).strip()[:600] + "\n\n")
    body.append(f"{comparison_label}\n", style="bold green")
    body.append(str(after).strip()[:600])
    console.print(Panel(body, title=mode, subtitle=chosen.id[:12]))